# 04 — Sales & Revenue Analysis

Olist E-Commerce Python Analytics

Covers: Overall Sales, Sales Over Time, Payments, Category Performance.
(Customer & Seller Intelligence moved out to notebook 05.)

## Setup

Load the cleaned tables plus the two lookup tables built during cleaning (`geo_lookup`, `order_level_reviews`).

In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv('../data/cleaned/olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
orders = pd.read_csv('../data/cleaned/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/cleaned/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/cleaned/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/cleaned/olist_order_reviews_dataset.csv')
order_level_reviews = pd.read_csv('../data/cleaned/order_level_reviews.csv')
products = pd.read_csv('../data/cleaned/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/cleaned/product_category_name_translation.csv')
sellers = pd.read_csv('../data/cleaned/olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})
geo_lookup = pd.read_csv('../data/cleaned/geo_lookup.csv', dtype={'geolocation_zip_code_prefix': str})


## Section 1 — Overall Sales

**Decision:** "revenue" = `item_total_value` (price + freight) throughout. Reported as GROSS (all orders) and NET (delivered only) pairs, never blended.

In [2]:
# Q1. How many orders are there?
totalOrders = orders['order_id'].count()
print(totalOrders)
# Finding: this is the denominator for every gross "average per order" figure below.


99441


In [3]:
# Q2. What is total item sales value? (gross — all statuses)
grossItemSales = order_items['price'].sum()
print(grossItemSales)

# Q3. What is total freight value? (gross)
grossFreightValue = order_items['freight_value'].sum()
print(grossFreightValue)

# Q4. What is total item + freight value? (gross)
grossItemFreightVal = order_items['item_total_value'].sum()
print(grossItemFreightVal)
# Finding: these three gross totals are the baseline every later filtered figure (net, category, seller) is compared back against.


13591643.7
2251909.54
15843553.24


In [4]:
# Q5. Average order value (gross). Orders with no items (775 of them)
#     contribute 0 to the numerator but still count in the denominator —
#     this is "average per order placed", not "average per order with items".
avgAllOrderValue = grossItemFreightVal / totalOrders
print(avgAllOrderValue)

# Q6. Average item value per order (gross)
avgAllItemValue = grossItemSales / totalOrders
print(avgAllItemValue)

# Q7. Average freight value per order (gross)
avgAllFreigthValue = grossFreightValue / totalOrders
print(avgAllFreigthValue)
# Finding: averages are deliberately "per order placed" — the 775 no-item orders dilute these slightly, by design, rather than being dropped.


159.32616566607336
136.68048088816482
22.64568477790851


In [5]:
# Q8. How many unique customers are there?
#     customer_unique_id, not customer_id — per the raw audit, customer_id
#     is a per-order surrogate key, not the real person.
print(customers['customer_unique_id'].nunique())
# Finding: customer_unique_id is the correct person-level key everywhere in this and the 05 notebook; customer_id would overcount by treating each order as a different person.


96096


In [6]:
# Net figures: restricted to delivered orders only, numerator and
# denominator from the same (delivered-only) population throughout.
deliveredOrders = orders[orders['order_status'] == 'delivered']
deliveredItems = order_items[order_items['order_id'].isin(deliveredOrders['order_id'])]

# Q9. How many orders were actually delivered? (net order count)
netTotalOrders = deliveredOrders['order_id'].count()
print(netTotalOrders)

# Q10. Net item sales value
netItemSales = deliveredItems['price'].sum()
print(netItemSales)

# Q11. Net freight value
netFreightValue = deliveredItems['freight_value'].sum()
print(netFreightValue)

# Q12. Net item + freight value
netItemFreightVal = deliveredItems['item_total_value'].sum()
print(netItemFreightVal)
# Finding: netTotalOrders / deliveredItems come from the same delivered-only population — avoids the mismatched-population trap (filtered revenue over an unfiltered order count) flagged repeatedly in earlier review passes.


96478
13221498.11
2198275.64
15419773.75


In [7]:
# Q13. Net average order value
avgNetOrderValue = netItemFreightVal / netTotalOrders
print(avgNetOrderValue)

# Q14. Net average item value per order
avgNetItemValue = netItemSales / netTotalOrders
print(avgNetItemValue)
# Finding: gross vs. net is reported as a labeled pair here, not blended — this convention is carried through every later section.


159.82683876116835
137.0415857501192


## Section 2 — Sales Over Time

**Decision:** time grouping is always on `order_purchase_timestamp` (when the customer ordered), never ship/delivery dates.

In [48]:
order_date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                    'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col])

order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

orders['purchaseMonth'] = orders['order_purchase_timestamp'].dt.to_period('M')
deliveredOrders = orders[orders['order_status'] == 'delivered']
deliveredItems = order_items[order_items['order_id'].isin(deliveredOrders['order_id'])]

# Merge all orders with their items (left join keeps the 775 no-item orders)
orderDetails = orders.merge(order_items, on='order_id', how='left')

# Merge only delivered orders with their items
deliveredOrderDetails = deliveredOrders.merge(deliveredItems, on='order_id', how='left')


In [49]:
# Q1. How do orders change over time? (gross)
orderOverTime = orders.groupby('purchaseMonth')['order_id'].count()
print(orderOverTime)

# Q1b. How do delivered orders change over time? (net)
deliveredOrderOverTime = deliveredOrders.groupby('purchaseMonth')['order_id'].count()
print(deliveredOrderOverTime)
# Finding: both series show the marketplace's 2016-09/10 ramp-up at the start — near-zero volume, not representative of steady-state behavior.


purchaseMonth
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: order_id, dtype: int64
purchaseMonth
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M, Name: order_id, dtype: int64


In [50]:
# Q2. Net (delivered) revenue over time
salesOverTime = deliveredOrderDetails.groupby('purchaseMonth')['item_total_value'].sum()
print(salesOverTime)
# Finding: this net monthly series is the primary revenue-over-time figure used by the growth calculations (Q8) below.


purchaseMonth
2016-09        143.46
2016-10      46490.66
2016-12         19.62
2017-01     127482.37
2017-02     271239.32
2017-03     414330.95
2017-04     390812.40
2017-05     566851.40
2017-06     490050.37
2017-07     566299.08
2017-08     645832.36
2017-09     701077.49
2017-10     751117.01
2017-11    1153364.20
2017-12     843078.29
2018-01    1077887.46
2018-02     966168.41
2018-03    1120598.24
2018-04    1132878.93
2018-05    1128774.52
2018-06    1011978.29
2018-07    1027807.28
2018-08     985491.64
Freq: M, Name: item_total_value, dtype: float64


In [51]:
# Q3. Gross monthly item sales
grossmonthlyItemSales = orderDetails.groupby('purchaseMonth')['price'].sum()
print(grossmonthlyItemSales)

# Q3b. Net monthly item sales
netmonthlyItemSales = deliveredOrderDetails.groupby('purchaseMonth')['price'].sum()
print(netmonthlyItemSales)

# Q4. Gross monthly freight
grossmonthlyFreightSales = orderDetails.groupby('purchaseMonth')['freight_value'].sum()
print(grossmonthlyFreightSales)

# Q4b. Net monthly freight
netmonthlyFreightSales = deliveredOrderDetails.groupby('purchaseMonth')['freight_value'].sum()
print(netmonthlyFreightSales)

# Q5. Gross monthly item + freight
grossmonthlyItemFreightSales = orderDetails.groupby('purchaseMonth')['item_total_value'].sum()
print(grossmonthlyItemFreightSales)

# Q5b. Net monthly item + freight
netmonthlyItemFreightSales = deliveredOrderDetails.groupby('purchaseMonth')['item_total_value'].sum()
print(netmonthlyItemFreightSales)
# Finding: gross/net pairs computed in parallel for item, freight, and item+freight — keeps the cancellation/non-fulfillment gap visible instead of hiding it inside one blended total.


purchaseMonth
2016-09        267.36
2016-10      49507.66
2016-12         10.90
2017-01     120312.87
2017-02     247303.02
2017-03     374344.30
2017-04     359927.23
2017-05     506071.14
2017-06     433038.60
2017-07     498031.48
2017-08     573971.68
2017-09     624401.69
2017-10     664219.43
2017-11    1010271.37
2017-12     743914.17
2018-01     950030.36
2018-02     844178.71
2018-03     983213.44
2018-04     996647.75
2018-05     996517.68
2018-06     865124.31
2018-07     895507.22
2018-08     854686.33
2018-09        145.00
2018-10          0.00
Freq: M, Name: price, dtype: float64
purchaseMonth
2016-09       134.97
2016-10     40325.11
2016-12        10.90
2017-01    111798.36
2017-02    234223.40
2017-03    359198.85
2017-04    340669.68
2017-05    489338.25
2017-06    421923.37
2017-07    481604.52
2017-08    554699.70
2017-09    607399.67
2017-10    648247.65
2017-11    987765.37
2017-12    726033.19
2018-01    924645.00
2018-02    826437.13
2018-03    953356.25
2018-04

In [52]:
# Q6. Gross average order value per month
grossmonthlyAvgOrderVal = grossmonthlyItemFreightSales / orderOverTime
print(grossmonthlyAvgOrderVal)

# Q6b. Net average order value per month
#      (denominator = deliveredOrderOverTime, an order-level count — not an
#      item-level row count, which would be inflated by multi-item orders)
netmonthlyAvgOrderVal = netmonthlyItemFreightSales / deliveredOrderOverTime
print(netmonthlyAvgOrderVal)
# Finding: net average divides by the delivered order-level count, not an item-row count — avoids double-counting multi-item orders, the same population-mismatch bug caught elsewhere in this project.


purchaseMonth
2016-09     88.687500
2016-10    175.335926
2016-12     19.620000
2017-01    171.485613
2017-02    160.831809
2017-03    161.091943
2017-04    171.556672
2017-05    158.429986
2017-06    154.996314
2017-07    145.298465
2017-08    154.284138
2017-09    168.121099
2017-10    166.122300
2017-11    156.302196
2017-12    152.220559
2018-01    152.332080
2018-02    146.686825
2018-03    160.189547
2018-04    167.127546
2018-05    167.289658
2018-06    165.830568
2018-07    168.265739
2018-08    154.070711
2018-09     10.403750
2018-10      0.000000
Freq: M, dtype: float64
purchaseMonth
2016-09    143.460000
2016-10    175.436453
2016-12     19.620000
2017-01    169.976493
2017-02    164.089123
2017-03    162.738001
2017-04    169.697091
2017-05    159.856571
2017-06    156.315907
2017-07    146.254928
2017-08    154.026320
2017-09    168.934335
2017-10    167.734929
2017-11    158.233530
2017-12    152.925502
2018-01    152.480897
2018-02    147.394113
2018-03    160.016884
20

In [53]:
# Q7. Month-over-month % change in order count (gross), reindexed across the
#     full date range so months with zero orders show as 0 rather than
#     being silently absent from the series.
months = pd.period_range(start='2016-09', end='2018-10', freq='M')
combined = orderOverTime.index.union(months)
orderOverTime_re = orderOverTime.reindex(combined)
orderOverTime_re = orderOverTime_re.fillna(0)
print(orderOverTime_re.pct_change().replace([np.inf, -np.inf], np.nan))
# Finding: reindexing across the full 2016-09–2018-10 range makes zero-order months show as a real 0 rather than silently dropping out of the series.


2016-09           NaN
2016-10     80.000000
2016-11     -1.000000
2016-12           NaN
2017-01    799.000000
2017-02      1.225000
2017-03      0.506742
2017-04     -0.103654
2017-05      0.539101
2017-06     -0.122973
2017-07      0.240678
2017-08      0.075758
2017-09     -0.010621
2017-10      0.080747
2017-11      0.629022
2017-12     -0.248012
2018-01      0.281333
2018-02     -0.074426
2018-03      0.071790
2018-04     -0.037720
2018-05     -0.009511
2018-06     -0.102721
2018-07      0.020269
2018-08      0.034965
2018-09     -0.997543
2018-10     -0.750000
Freq: M, Name: order_id, dtype: float64


In [54]:
# Q8. Month-over-month % change in net revenue (reindexed)
months = pd.period_range(start='2016-09', end='2018-10', freq='M')
combined = salesOverTime.index.union(months)
salesOverTime_re = salesOverTime.reindex(combined)
salesOverTime_re = salesOverTime_re.fillna(0)
print(salesOverTime_re.pct_change().replace([np.inf, -np.inf], np.nan))
# NOTE: 2016-09/10/12 and 2018-09/10 are near-empty ramp-up/tail-off months —
# pct_change() here reflects a near-zero base, not real volatility; exclude
# from any growth narrative.
# Finding: confirmed — the large/undefined pct_change spikes are a near-zero-base artifact in the edge months, not genuine volatility.


2016-09            NaN
2016-10     323.067057
2016-11      -1.000000
2016-12            NaN
2017-01    6496.572375
2017-02       1.127661
2017-03       0.527548
2017-04      -0.056763
2017-05       0.450444
2017-06      -0.135487
2017-07       0.155594
2017-08       0.140444
2017-09       0.085541
2017-10       0.071375
2017-11       0.535532
2017-12      -0.269027
2018-01       0.278514
2018-02      -0.103646
2018-03       0.159837
2018-04       0.010959
2018-05      -0.003623
2018-06      -0.103472
2018-07       0.015642
2018-08      -0.041171
2018-09      -1.000000
2018-10            NaN
Freq: M, Name: item_total_value, dtype: float64


In [55]:
print('purchaseMonth' in orders.columns)
print('purchaseMonth' in deliveredOrders.columns)
print('purchaseMonth' in deliveredItems.columns)
# Finding: sanity check confirming purchaseMonth propagated correctly to every frame used in the groupbys above before trusting their output.


True
True
False


## Section 3 — Payments

In [56]:
# Q1. Revenue split by payment type
paymentRevType = order_payments.groupby('payment_type')['payment_value'].sum()
paymentRevType.sort_values(ascending=False)
# Finding: credit card is ~79% of revenue — over-indexed relative to its share of transactions (see Q2/count below).


payment_type
credit_card    12542084.19
boleto          2869361.27
voucher          379436.87
debit_card       217989.79
not_defined           0.00
Name: payment_value, dtype: float64

In [57]:
# Q2. Average payment value by payment type
avgPaymentRevType = order_payments.groupby('payment_type')['payment_value'].mean()
avgPaymentRevType.sort_values(ascending=False)
# Finding: credit card also has the highest average value (~₹163.32), consistent with its outsized revenue share.


payment_type
credit_card    163.319021
boleto         145.034435
debit_card     142.570170
voucher         65.703354
not_defined      0.000000
Name: payment_value, dtype: float64

In [58]:
# Transaction count by type (for comparing revenue share vs. volume share)
PaymentRevTypeCount = order_payments.groupby('payment_type')['payment_value'].count()
PaymentRevTypeCount.sort_values(ascending=False)
# Finding: credit card's ~79% revenue share outpaces its ~74% transaction share — it's genuinely over-indexed, not just more frequent.


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: payment_value, dtype: int64

In [59]:
order_payments.columns


Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'zero_value_flag'],
      dtype='str')

In [60]:
# Q3. Does installment count correlate with payment value? Run at the ROW
#     level within credit_card only (76,795 rows) — the only payment type
#     where installments genuinely vary; correlating 5 category averages
#     against each other would be a far weaker, less meaningful signal.
creditCardPayments = order_payments[order_payments['payment_type'] == 'credit_card']
print(creditCardPayments[['payment_installments', 'payment_value']].corr())
# Finding: r ≈ 0.38 — a real but moderate positive relationship between installment count and payment value.


                      payment_installments  payment_value
payment_installments              1.000000       0.376499
payment_value                     0.376499       1.000000


In [61]:
# Q4. Installments by payment type — confirms installments are a
#     credit-card-specific feature; every other type sits at exactly 1.0.
a = order_payments.groupby('payment_type').agg(
    avgPay=('payment_value', 'mean'),
    avgInstalment=('payment_installments', 'mean')
)
a
# Finding: confirmed — every non-credit-card payment type averages exactly 1.0 installments; installments are a credit-card-only feature.


,avgPay,avgInstalment
payment_type,,
boleto,145.034435,1.000000
credit_card,163.319021,3.507181
debit_card,142.570170,1.000000
not_defined,0.000000,1.000000
voucher,65.703354,1.000000


In [62]:
orderWithPayments = order_items.merge(order_payments, on='order_id', how='left')
orderWithPayments.columns


Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'priceFreightCheck',
       'item_total_value', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'zero_value_flag'],
      dtype='str')

In [63]:
# Q5. Do per-order payment totals reconcile with per-order item totals?
#     Built as a merged DataFrame (not a raw Series != Series comparison,
#     which misaligns on index and gives unreliable results).
paymentTotals = order_payments.groupby('order_id')['payment_value'].sum()
itemTotals = order_items.groupby('order_id')['item_total_value'].sum()

reconciliation = pd.DataFrame({
    'paymentTotal': paymentTotals,
    'itemTotal': itemTotals
})

reconciliation['mismatch'] = abs(reconciliation['paymentTotal']) != abs(reconciliation['itemTotal'])
print(reconciliation['mismatch'].sum())
# Finding: 8,827 raw mismatches — most of this is NaN-driven (see below), not genuine value disagreement.


8827


In [64]:
# Filter out floating-point noise (cent-level rounding) to find genuine mismatches
reconciliation['diff'] = reconciliation['paymentTotal'] - reconciliation['itemTotal']
realMismatch = reconciliation[reconciliation['diff'].abs() > 0.01]
print(realMismatch.shape[0])
# Finding: only 375 of the 8,827 raw mismatches are genuine (>₹0.01) value discrepancies.


375


In [65]:
# How many raw mismatches are NaN-driven (an order missing from one side
# entirely — the 1 no-payment order / 775 no-item orders)?
print(reconciliation['paymentTotal'].isna().sum(), reconciliation['itemTotal'].isna().sum())
# Finding: 776 of the raw mismatches are NaN-driven — the 1 no-payment order plus the 775 no-item orders — leaving 375 as the genuine mismatch count.


1 775


In [66]:
# Direction of the genuine mismatches — overpaid or underpaid relative to items?
print(realMismatch['diff'].describe())
# Finding: genuine mismatches are mostly small overpayments — mean ~₹7.66, median ~₹3.23.


count    375.000000
mean       7.655520
std       16.668871
min      -51.620000
25%        0.010000
50%        3.230000
75%        9.700000
max      182.810000
Name: diff, dtype: float64


In [67]:
# Does the mismatch concentrate in particular months? (raw count — see rate below)
mismatchWithTime = realMismatch.merge(orders[['order_id', 'purchaseMonth']], on='order_id', how='left')
print(mismatchWithTime.groupby('purchaseMonth').size())


purchaseMonth
2016-10     5
2017-01     8
2017-02    13
2017-03     5
2017-04    13
2017-05    23
2017-06    15
2017-07    14
2017-08     8
2017-09    11
2017-10    16
2017-11    38
2017-12    21
2018-01    48
2018-02    33
2018-03    15
2018-04    18
2018-05    12
2018-06     8
2018-07    15
2018-08    36
Freq: M, dtype: int64


In [68]:
# Mismatch RATE (normalized by that month's total order volume) — the raw
# count above is misleading on its own since order volume itself grows
# heavily across the date range.
mismatchRate = mismatchWithTime.groupby('purchaseMonth').size() / orderOverTime * 100
print(mismatchRate)
# Finding: mismatch rate is elevated in the platform's earliest, lowest-volume months (2016-10, 2017-01 — both >1%), settling into a stable 0.2%–0.6% band thereafter. The raw-count spikes in Nov 2017/Aug 2018 are volume artifacts, not a rate increase.


purchaseMonth
2016-09         NaN
2016-10    1.543210
2016-12         NaN
2017-01    1.000000
2017-02    0.730337
2017-03    0.186428
2017-04    0.540765
2017-05    0.621622
2017-06    0.462250
2017-07    0.347740
2017-08    0.184715
2017-09    0.256709
2017-10    0.345498
2017-11    0.503712
2017-12    0.370175
2018-01    0.660338
2018-02    0.490488
2018-03    0.208016
2018-04    0.259403
2018-05    0.174596
2018-06    0.129723
2018-07    0.238398
2018-08    0.552826
2018-09         NaN
2018-10         NaN
Freq: M, dtype: float64


In [69]:
# Q6. How does revenue by payment type trend month over month?
paymentTypeMonthly = orderWithPayments.merge(
    orders[['order_id', 'purchaseMonth']], on='order_id', how='left'
).groupby(['purchaseMonth', 'payment_type']).agg(
    count=('payment_type', 'size'),
    revenue=('payment_value', 'sum')
).unstack()
print(paymentTypeMonthly)


                count                                   revenue              \
payment_type   boleto credit_card debit_card voucher     boleto credit_card   
purchaseMonth                                                                 
2016-09           NaN         3.0        NaN     NaN        NaN      347.52   
2016-10          71.0       286.0        2.0    22.0   13981.39    58874.38   
2016-12           NaN         1.0        NaN     NaN        NaN       19.62   
2017-01         232.0       695.0       10.0    68.0   33013.73   148538.18   
2017-02         438.0      1488.0       14.0   122.0   72366.68   263399.31   
2017-03         667.0      2252.0       36.0   226.0   96089.44   414763.88   
2017-04         567.0      2053.0       28.0   207.0  120183.98   370349.16   
2017-05         870.0      3184.0       38.0   304.0  174185.54   528616.15   
2017-06         810.0      2697.0       29.0   245.0  145956.20   438653.30   
2017-07         970.0      3452.0       23.0   395.0

In [70]:
# Q7. Confirm the zero-value not_defined rows are exactly the 3 Broken
#     Record rows identified during cleaning.
print(order_payments[order_payments['payment_type'] == 'not_defined'])
# Finding: confirmed — the 3 not_defined/Broken Record rows (₹0 value) match the 3 identified during cleaning; excluded from any meaningful revenue-share conclusion.


                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value zero_value_flag  
51280                     1            0.0   Broken Record  
57411                     1            0.0   Broken Record  
94427                     1            0.0   Broken Record  


In [71]:
# Has credit card's revenue share shifted over time, or stayed stable?
monthlyRevenue = paymentTypeMonthly['revenue']
creditCardShare = monthlyRevenue['credit_card'] / monthlyRevenue.sum(axis=1) * 100
print(creditCardShare)
# Finding: credit card's revenue share is stable throughout (~75%–81%, excluding near-empty edge months) — no growing or shrinking trend.


purchaseMonth
2016-09    100.000000
2016-10     79.651917
2016-12    100.000000
2017-01     79.102485
2017-02     76.539576
2017-03     78.708550
2017-04     73.239946
2017-05     72.962433
2017-06     73.017214
2017-07     75.103343
2017-08     77.616551
2017-09     78.779025
2017-10     77.129991
2017-11     76.395246
2017-12     75.917282
2018-01     76.443947
2018-02     74.872828
2018-03     79.296583
2018-04     77.232867
2018-05     77.785384
2018-06     79.476696
2018-07     72.307713
2018-08     80.127321
2018-09           NaN
Freq: M, dtype: float64


## Section 4 — Category Performance

**Decision:** GROSS figures only — this section measures demand ("what are people trying to buy"), not fulfillment.

In [72]:
# Base table: every sold item joined to its product/category info.
# Anchored on order_items (one row per sale) with products left-joined on —
# never the other way around, which would inflate counts with unsold catalog items.
orderItemsWithCategory = order_items.merge(products, on='product_id', how='left')
orderItemsWithCategory.columns


Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'priceFreightCheck',
       'item_total_value', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'listing_metadata_missing', 'dimensions_missing',
       'product_category_name_english'],
      dtype='str')

In [73]:
# Q1a. Top categories by item-sold count
productCategoriesCount = orderItemsWithCategory.groupby('product_category_name')['product_category_name'].count()
print(productCategoriesCount.sort_values(ascending=False).head(20))

# Q1b. Top categories by gross revenue
productCategoriesGrossRev = orderItemsWithCategory.groupby('product_category_name')['item_total_value'].sum()

# (price-only revenue, used as denominator for freight-ratio work below)
productCategoriesItemsRev = orderItemsWithCategory.groupby('product_category_name')['price'].sum()

print(productCategoriesGrossRev.sort_values(ascending=False).head(10))
print(productCategoriesItemsRev.sort_values(ascending=False).head(10))
# Finding: rankings diverge between the two views — cama_mesa_banho leads on item count, while beleza_saude/relogios_presentes lead on revenue.


product_category_name
cama_mesa_banho                11115
beleza_saude                    9670
esporte_lazer                   8641
moveis_decoracao                8334
informatica_acessorios          7827
utilidades_domesticas           6964
relogios_presentes              5991
telefonia                       4545
ferramentas_jardim              4347
automotivo                      4235
brinquedos                      4117
cool_stuff                      3796
perfumaria                      3419
bebes                           3065
eletronicos                     2767
papelaria                       2517
fashion_bolsas_e_acessorios     2031
pet_shop                        1947
moveis_escritorio               1691
consoles_games                  1137
Name: product_category_name, dtype: int64
product_category_name
beleza_saude              1441248.07
relogios_presentes        1305541.61
cama_mesa_banho           1241681.72
esporte_lazer             1156656.48
informatica_acessorios    

In [74]:
# Q2. Average item price by category (revenue ÷ that category's own item
#     count — both from the same merged, item-level table)
productCategoriesGrossAvg = productCategoriesGrossRev / productCategoriesCount
print(productCategoriesGrossAvg.sort_values(ascending=False).head(10))


product_category_name
pcs                                              1146.795222
portateis_casa_forno_e_cafe                       660.441711
eletrodomesticos_2                                520.663529
agro_industria_e_comercio                         369.689009
instrumentos_musicais                             309.025544
eletroportateis                                   304.372356
portateis_cozinha_e_preparadores_de_alimentos     285.219333
telefonia_fixa                                    243.260644
construcao_ferramentas_seguranca                  229.193918
moveis_quarto                                     226.247798
dtype: float64


In [75]:
# Q3. Freight ratio per category = freight ÷ price (NOT ÷ item_total_value,
#     which already includes freight and caps the ratio under 1.0, hiding
#     categories where freight genuinely exceeds price).
freightCategory = orderItemsWithCategory.groupby('product_category_name')['freight_value'].sum()
avgFreightRatioCategory = freightCategory / productCategoriesItemsRev
avgFreightRatioCategory.sort_values(ascending=False).head(10)
# Finding: no category averages above 1.0 — freight never exceeds price on a category-wide basis, even though individual flagged items do (see Q4).


product_category_name
casa_conforto_2              0.539690
flores                       0.440408
moveis_colchao_e_estofado    0.373267
artigos_de_natal             0.366932
fraldas_higiene              0.365963
cds_dvds_musicais            0.308205
sinalizacao_e_seguranca      0.302559
alimentos_bebidas            0.296979
eletronicos                  0.290666
fashion_esporte              0.272766
dtype: float64

In [76]:
# Q4. What % of total revenue comes from "Delivery Expensive" items
#     (freight > price, flagged during cleaning)?
expensiveDeliveries = orderItemsWithCategory[orderItemsWithCategory['freight_value'] > orderItemsWithCategory['price']]
expensiveDeliveryRevenue = expensiveDeliveries['item_total_value'].sum()
totalGrossRevenue = orderItemsWithCategory['item_total_value'].sum()
print(expensiveDeliveryRevenue / totalGrossRevenue * 100)
# Finding: ~1.09% of total revenue despite being ~3.7% of transactions — confirms these are low-value items, not high-value ones.


1.0920995901548154


In [77]:
# Q4b. Which categories concentrate these items, and at what RATE
#      (normalized by that category's own item count — a raw count favors
#      high-volume categories regardless of true freight severity)?
deliveryExpensiveByCategory = expensiveDeliveries.groupby('product_category_name')['freight_value'].count()
print(deliveryExpensiveByCategory.sort_values(ascending=False).head(10))

deliveryExpensiveRate = deliveryExpensiveByCategory / productCategoriesCount
print(deliveryExpensiveRate.sort_values(ascending=False).head(10))
# Finding: casa_conforto_2 leads both the freight-ratio ranking (Q3) and this rate ranking — a consistent, well-supported finding.


product_category_name
eletronicos               611
telefonia                 430
utilidades_domesticas     385
moveis_decoracao          334
beleza_saude              299
esporte_lazer             221
cama_mesa_banho           204
automotivo                174
informatica_acessorios    174
perfumaria                144
Name: freight_value, dtype: int64
product_category_name
casa_conforto_2                   0.533333
dvds_blu_ray                      0.375000
artigos_de_natal                  0.222222
eletronicos                       0.220817
artes_e_artesanato                0.125000
fashion_underwear_e_moda_praia    0.122137
sinalizacao_e_seguranca           0.120603
musica                            0.105263
audio                             0.101648
telefonia_fixa                    0.094697
dtype: float64


In [78]:
# Q5. Do products with completely missing listing metadata
#     (listing_metadata_missing == True) still generate real revenue?
missingMetaDeta = orderItemsWithCategory[orderItemsWithCategory['listing_metadata_missing'] == True]
missingMetaDetaRevenue = missingMetaDeta['item_total_value'].sum()
print(missingMetaDetaRevenue / totalGrossRevenue * 100)
print(missingMetaDeta.shape[0], missingMetaDeta['product_id'].nunique())
# Finding: ~1.31% of total revenue across 1,603 sales from 610 distinct products — a real, non-trivial data-quality finding.


1.3109754286406528
1603 610


In [79]:
# Q6. Category revenue concentration at a few cutoffs
top10Gross = productCategoriesGrossRev.sort_values(ascending=False).head(10).sum()
top10Gross / totalGrossRevenue * 100

for n in [3, 5, 10, 20]:
    share = productCategoriesGrossRev.sort_values(ascending=False).head(n).sum() / totalGrossRevenue * 100
    print(n, share)
# Finding: top 3/5/10/20 categories = 25.2% / 39.2% / 62.3% / 83.9% of total revenue — a steep Pareto curve.


3 25.174096615715985
5 39.16040919618863
10 62.3234142646148
20 83.86924680792123


In [80]:
# Q7. How many unique sellers operate in each category?
sellersPerCategory = orderItemsWithCategory.groupby('product_category_name')['seller_id'].nunique()
print(sellersPerCategory.sort_values(ascending=False).head(10))


product_category_name
beleza_saude              492
esporte_lazer             481
utilidades_domesticas     468
automotivo                383
moveis_decoracao          370
informatica_acessorios    287
cool_stuff                267
brinquedos                252
bebes                     244
ferramentas_jardim        237
Name: seller_id, dtype: int64


In [81]:
# Q8. How does revenue for the top categories trend month over month?
orderItemsWithCategoryTime = orderItemsWithCategory.merge(
    orders[['order_id', 'purchaseMonth']], on='order_id', how='left'
)

top10Cat = productCategoriesGrossRev.sort_values(ascending=False).head(10).index
filtered = orderItemsWithCategoryTime[orderItemsWithCategoryTime['product_category_name'].isin(top10Cat)]
categoryMonthlyRevenue = filtered.groupby(['purchaseMonth', 'product_category_name'])['item_total_value'].sum().unstack()
categoryMonthlyRevenue


product_category_name,automotivo,beleza_saude,cama_mesa_banho,cool_stuff,esporte_lazer,ferramentas_jardim,informatica_acessorios,moveis_decoracao,relogios_presentes,utilidades_domesticas
purchaseMonth,,,,,,,,,,
2016-09,NaN,143.46,NaN,NaN,NaN,NaN,NaN,136.23,NaN,NaN
2016-10,2257.56,5493.38,606.58,1327.11,3927.11,1465.19,1740.09,7052.28,3468.04,1542.54
2017-01,5844.21,14039.74,4817.27,8730.86,11447.27,9225.17,4377.60,16461.75,8310.83,3269.76
2017-02,15195.67,26130.08,19297.79,14857.05,19389.03,14745.30,13744.35,26262.32,12370.67,14207.49
2017-03,16755.73,29852.87,30650.01,23389.43,31309.60,27004.32,31873.84,29506.43,28066.55,17180.85
2017-04,17799.93,26397.99,29007.36,24393.33,29121.39,23468.07,23180.46,26128.42,25153.31,18356.26
2017-05,21631.70,52801.02,39705.14,38518.39,42376.95,24100.00,51306.32,29971.21,40377.04,27045.60
2017-06,33693.74,37276.48,41684.51,42730.63,39043.92,18596.95,41621.82,22017.99,30918.54,29857.93
2017-07,16490.82,40419.76,74629.84,42328.60,46324.06,24251.57,44599.21,30630.86,39585.76,25596.16


In [82]:
# Q9. Does freight cost scale with product weight, or are some categories
#     freight-expensive despite being light (a pricing-structure issue, not
#     a physical-bulk one)?
categoryWeightFreight = orderItemsWithCategory.groupby('product_category_name').agg(
    avgWeight=('product_weight_g', 'mean'),
    avgFreight=('freight_value', 'mean')
)
print(categoryWeightFreight.sort_values('avgFreight', ascending=False).head(10))

print(categoryWeightFreight.loc[['casa_conforto_2', 'flores', 'artigos_de_natal', 'fraldas_higiene']])
# Finding: the highest-absolute-freight categories are furniture/appliances — genuinely heavy, expected. The highest-freight-ratio categories (casa_conforto_2, flores, etc.) are comparatively light — their freight burden is a pricing-structure issue (cheap item, ordinary shipping fee), not a physical-bulk one.


                                                   avgWeight  avgFreight
product_category_name                                                   
pcs                                              7216.798030   48.454680
eletrodomesticos_2                               8957.571429   44.538571
moveis_colchao_e_estofado                        7557.894737   42.906842
moveis_cozinha_area_de_servico_jantar_e_jardim   8852.850534   42.702598
moveis_quarto                                    9989.220183   42.497523
moveis_escritorio                               11390.454169   40.551124
portateis_casa_forno_e_cafe                      5074.894737   36.156053
moveis_sala                                      8092.532803   35.722008
sinalizacao_e_seguranca                          3886.723618   32.702613
industria_comercio_e_negocios                    6663.652985   29.420448
                         avgWeight  avgFreight
product_category_name                         
casa_conforto_2         827.93

In [83]:
# Q10. Is the average price per category a reliable "typical price", or
#      skewed by a long tail of pricier items? (mean vs. median gap)
top10Cat = productCategoriesGrossRev.sort_values(ascending=False).head(10).index
filtered = orderItemsWithCategoryTime[orderItemsWithCategoryTime['product_category_name'].isin(top10Cat)]
result = filtered.groupby('product_category_name').agg(
    meanPrice=('price', 'mean'),
    medianPrice=('price', 'median')
)
result['pctGap'] = (result['meanPrice'] - result['medianPrice']) / result['medianPrice'] * 100
print(result.sort_values('pctGap', ascending=False))
# Finding: every one of the top 10 categories shows mean meaningfully above median (18%–86% gap) — right-skewed pricing is the norm, not the exception, so median (or both figures together) is the more honest "typical price" to report.


                         meanPrice  medianPrice     pctGap
product_category_name                                     
ferramentas_jardim      111.630196        59.90  86.360927
automotivo              139.957523        84.90  64.849850
beleza_saude            130.163531        79.90  62.908048
relogios_presentes      201.135984       129.00  55.919367
utilidades_domesticas    90.788148        59.80  51.819645
esporte_lazer           114.344285        78.00  46.595238
informatica_acessorios  116.513903        81.99  42.107456
moveis_decoracao         87.564494        65.49  33.706663
cool_stuff              167.357969       129.99  28.746803
cama_mesa_banho          93.296327        79.05  18.021920
